# LeetCode 1625 — Lexicographically Smallest String After Applying Operations bằng Python

Link đề: https://leetcode.com/problems/lexicographically-smallest-string-after-applying-operations/ · Độ khó: **Medium** · Chủ đề: String, Depth-First Search, Breadth-First Search, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def findLexSmallestString(self, s: str, a: int, b: int) -> str:
        ...
```

## 1. Mục tiêu học

1. Coi mỗi chuỗi là một **trạng thái** và mỗi phép toán là một **cạnh** ⇒ dùng **BFS/DFS + tập `visited`** để duyệt toàn bộ không gian trạng thái hữu hạn.
2. Phân tích cấu trúc của phép xoay: các độ xoay đạt được là **bội của `gcd(b, n)`**; và **tính chẵn lẻ của `b`** quyết định có cộng được vào các chỉ số chẵn hay không.
3. Đi từ BFS trên chuỗi → **liệt kê có hệ thống** `(độ xoay, số lần cộng lẻ, số lần cộng chẵn)` → **tham lam** theo chữ số đầu tiên.
4. Dùng thành thạo so sánh chuỗi theo thứ tự từ điển trong Python (`<`, `min`), cắt chuỗi để xoay (`s[-b:] + s[:-b]`) và số học modulo 10.
5. Nhận ra các bẫy: chỉ thử xoay `b` một lần, quên cộng vào chỉ số chẵn khi `b` lẻ, lặp cộng thiếu lần, BFS quên `visited`.

## 2. Đề bài & ví dụ

Cho chuỗi `s` có **độ dài chẵn**, chỉ gồm chữ số `0`–`9`, và hai số nguyên `a`, `b`. Bạn được áp dụng **bao nhiêu lần tùy ý, theo thứ tự tùy ý** hai phép:
1. **Cộng `a`** vào **mọi chỉ số lẻ** (đánh số từ 0: chỉ số `1, 3, 5, …`). Chữ số vượt 9 quay vòng về 0 (tức là cộng modulo 10). Ví dụ `s = "3456"`, `a = 5` ⇒ `"3951"`.
2. **Xoay phải** `s` đi `b` vị trí. Ví dụ `s = "3456"`, `b = 1` ⇒ `"6345"`.

Trả về chuỗi **nhỏ nhất theo thứ tự từ điển** có thể thu được.

**Ví dụ 1:** `s = "5525", a = 9, b = 2` → `"2050"` (xoay → `"2555"`, cộng → `"2454"`, cộng → `"2353"`, xoay → `"5323"`, … cuối cùng `"2050"`).

**Ví dụ 2:** `s = "74", a = 5, b = 1` → `"24"` (xoay `"47"`, cộng `"42"`, xoay `"24"`).

**Ví dụ 3:** `s = "0011", a = 4, b = 2` → `"0011"` (không làm nhỏ hơn được).

**Ràng buộc:**
- `2 ≤ s.length ≤ 100`, `s.length` chẵn
- `s` chỉ gồm chữ số
- `1 ≤ a ≤ 9`
- `1 ≤ b ≤ s.length − 1`

In [1]:
import random
import time
from collections import deque
from math import gcd

examples = [
    (("5525", 9, 2), "2050"),
    (("74", 5, 1), "24"),
    (("0011", 4, 2), "0011"),
]
for (s, a, b), want in examples:
    print(f"s={s!r}, a={a}, b={b} -> kỳ vọng {want!r}")

s='5525', a=9, b=2 -> kỳ vọng '2050'
s='74', a=5, b=1 -> kỳ vọng '24'
s='0011', a=4, b=2 -> kỳ vọng '0011'


## 3. Kiến thức nền

### 3.1 So sánh chuỗi theo thứ tự từ điển

Python so sánh chuỗi **từng ký tự từ trái sang phải**, ký tự đầu tiên khác nhau quyết định. Với hai chuỗi chữ số **cùng độ dài**, đây đúng là thứ tự đề yêu cầu. Hệ quả quan trọng: **chữ số đầu tiên quan trọng hơn tất cả các chữ số phía sau cộng lại**.

In [2]:
print("0158" < "0190", "2050" < "2454", min(["5525", "2555", "2050"]))
print("chữ số đầu quyết định:", "1999" < "2000")

True True 2050
chữ số đầu quyết định: True


### 3.2 Xoay phải bằng cắt chuỗi

Xoay phải `b` vị trí: `b` ký tự cuối lên đầu ⇒ `s[-b:] + s[:-b]`. Xoay `b` rồi xoay `c` = xoay `(b + c) mod n`. Một cách tổng quát, ký tự ở vị trí `i` đi tới vị trí `(i + b) mod n`.

In [3]:
def rotate(s, b):
    b %= len(s)
    return s[-b:] + s[:-b] if b else s

print(rotate("3456", 1), rotate("3456", 2), rotate(rotate("3456", 1), 2) == rotate("3456", 3))

6345 5634 True


### 3.3 Cộng modulo 10 vào các chỉ số lẻ

`(int(c) + a) % 10` cho chữ số mới. Áp dụng `k` lần ⇒ cộng `k·a mod 10`. Vì `10·a ≡ 0 (mod 10)` nên **chỉ cần thử `k = 0..9`**; thực ra chu kỳ là `10 / gcd(a, 10)`.

In [4]:
def add_odd(s, a):
    return "".join(str((int(c) + a) % 10) if i % 2 == 1 else c for i, c in enumerate(s))

print(add_odd("3456", 5))
for a in (1, 2, 4, 5, 9):
    shifts = sorted({k * a % 10 for k in range(10)})
    print(f"a={a}: các độ cộng đạt được {shifts}  (chu kỳ {10 // gcd(a, 10)})")

3951
a=1: các độ cộng đạt được [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  (chu kỳ 10)
a=2: các độ cộng đạt được [0, 2, 4, 6, 8]  (chu kỳ 5)
a=4: các độ cộng đạt được [0, 2, 4, 6, 8]  (chu kỳ 5)
a=5: các độ cộng đạt được [0, 5]  (chu kỳ 2)
a=9: các độ cộng đạt được [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  (chu kỳ 10)


### 3.4 BFS với `deque` và `set`

Duyệt đồ thị trạng thái: hàng đợi `deque` chứa trạng thái chờ xét, `visited` (một `set`) để không thăm lại ⇒ mỗi trạng thái xét đúng một lần, BFS chắc chắn dừng nếu số trạng thái hữu hạn.

In [5]:
def bfs_demo(start, neighbors):
    seen, q = {start}, deque([start])
    while q:
        x = q.popleft()
        for y in neighbors(x):
            if y not in seen:
                seen.add(y); q.append(y)
    return seen

# ví dụ: các số đạt được từ 0 bằng "+3 mod 12"
print(sorted(bfs_demo(0, lambda x: [(x + 3) % 12])))

[0, 3, 6, 9]


### 3.5 `math.gcd`

`gcd(b, n)`: ước chung lớn nhất. Tập `{k·b mod n : k ≥ 0}` đúng bằng các bội của `gcd(b, n)` trong `0..n−1` (tính chất nhóm cộng modulo `n`).

In [6]:
for n, b in [(4, 2), (6, 4), (8, 3), (10, 6)]:
    reach = sorted({k * b % n for k in range(n)})
    print(f"n={n}, b={b}: gcd={gcd(b, n)}, xoay đạt được {reach}")

n=4, b=2: gcd=2, xoay đạt được [0, 2]
n=6, b=4: gcd=2, xoay đạt được [0, 2, 4]
n=8, b=3: gcd=1, xoay đạt được [0, 1, 2, 3, 4, 5, 6, 7]
n=10, b=6: gcd=2, xoay đạt được [0, 2, 4, 6, 8]


## 4. Ý tưởng nền tảng / trực giác

Làm tay ví dụ 1 `s = "5525", a = 9, b = 2` (`n = 4`):
- Các độ xoay đạt được: bội của `gcd(2, 4) = 2` ⇒ `{0, 2}` ⇒ hai "khung" `"5525"` và `"2555"`.
- `b = 2` chẵn ⇒ chữ số ở chỉ số chẵn **không bao giờ** đổi sang vị trí lẻ ⇒ chỉ cộng được vào nhóm lẻ.
- Khung `"5525"`: chữ số đầu 5 (không đổi được). Khung `"2555"`: chữ số đầu 2 ⇒ khung này thắng.
- Trong khung `"2555"`, cộng `9` vào chỉ số lẻ `k` lần: chữ số ở chỉ số 1 là `(5 + 9k) mod 10`, nhỏ nhất bằng 0 khi `k = 5` ⇒ `"2050"` ✔.

Làm tay ví dụ 2 `s = "74", a = 5, b = 1`:
- `b = 1` lẻ ⇒ xoay 1 lần thì nhóm chẵn và nhóm lẻ đổi chỗ ⇒ **cả hai nhóm đều cộng được**, độc lập nhau.
- Khung `"74"`: chữ số đầu `7 → {7, 2}` ⇒ 2; chữ số sau `4 → {4, 9}` ⇒ 4 ⇒ `"24"`.
- Khung `"47"`: `4 → 4`, `7 → 2` ⇒ `"42"`. So sánh ⇒ `"24"` ✔.

In [7]:
s, a, b = "5525", 9, 2
for r in (0, 2):
    t = rotate(s, r)
    best = min("".join(str((int(c) + k * a) % 10) if i % 2 else c for i, c in enumerate(t)) for k in range(10))
    print(f"xoay {r}: khung {t!r} -> tốt nhất {best!r}")

xoay 0: khung '5525' -> tốt nhất '5020'
xoay 2: khung '2555' -> tốt nhất '2050'


## 5. Cách 1 — Brute Force / baseline: BFS trên mọi chuỗi đạt được

**Ý tưởng:** mỗi chuỗi là một đỉnh; từ mỗi đỉnh có 2 cạnh (cộng, xoay). Số chuỗi đạt được là hữu hạn (nhiều nhất `n · 10 · 10`), nên BFS với `visited` sẽ dừng. Trả về chuỗi nhỏ nhất trong `visited`.

In [8]:
def smallest_bfs(s, a, b):
    seen, q = {s}, deque([s])
    while q:
        x = q.popleft()
        for y in (add_odd(x, a), rotate(x, b)):
            if y not in seen:
                seen.add(y); q.append(y)
    return min(seen)

def reachable_bfs(s, a, b):
    seen, q = {s}, deque([s])
    while q:
        x = q.popleft()
        for y in (add_odd(x, a), rotate(x, b)):
            if y not in seen:
                seen.add(y); q.append(y)
    return seen

for (s, a, b), want in examples:
    print((s, a, b), smallest_bfs(s, a, b), "kỳ vọng", want, "| số trạng thái:", len(reachable_bfs(s, a, b)))

('5525', 9, 2) 2050 kỳ vọng 2050 | số trạng thái: 20
('74', 5, 1) 24 kỳ vọng 24 | số trạng thái: 8
('0011', 4, 2) 0011 kỳ vọng 0011 | số trạng thái: 10


**Độ phức tạp:** gọi `n = len(s)`. Số trạng thái `≤ n · 10 · 10 = 100n`; mỗi trạng thái tốn `O(n)` để tạo 2 chuỗi con và băm ⇒ Time `O(100·n²)` ≈ `10⁶` khi `n = 100`, Space `O(100·n²)` ký tự. BFS **đã được chấp nhận** trên LeetCode — nhưng nó không cho ta hiểu cấu trúc bài.

**Nút thắt:** lưu và băm tới `10⁴` chuỗi dài 100; và ta phải tin rằng không gian trạng thái nhỏ. Câu hỏi tự nhiên: *chính xác thì những chuỗi nào đạt được?* Trả lời được câu này sẽ dẫn tới lời giải liệt kê trực tiếp không cần `visited`.

In [9]:
for s, a, b in [("43987654", 7, 3), ("0123456789" * 10, 3, 7), ("0123456789" * 10, 5, 10)]:
    t = time.perf_counter()
    cnt = len(reachable_bfs(s, a, b))
    print(f"n={len(s):>3}, a={a}, b={b}: {cnt:>5} trạng thái, {time.perf_counter() - t:.3f}s")

n=  8, a=7, b=3:   800 trạng thái, 0.001s
n=100, a=3, b=7:   100 trạng thái, 0.001s
n=100, a=5, b=10:     2 trạng thái, 0.000s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — Phép xoay.** Xoay `b` lặp lại nhiều lần cho độ xoay `k·b mod n`; tập này chính là **bội của `g = gcd(b, n)`**. Có đúng `n / g` khung xoay.

**Insight 2 — Tính chẵn lẻ.** Ký tự ở vị trí `i` sau khi xoay `r` nằm ở `(i + r) mod n`. Vì `n` chẵn, tính chẵn lẻ của vị trí đổi **khi và chỉ khi `r` lẻ**.
- **Nếu `b` chẵn** thì mọi độ xoay đều chẵn ⇒ ký tự gốc ở vị trí chẵn mãi mãi ở vị trí chẵn ⇒ **không bao giờ được cộng**.
- **Nếu `b` lẻ** thì ta xoay lẻ, cộng (tác động lên nhóm gốc chẵn), rồi xoay tiếp cho đủ vòng ⇒ **cả hai nhóm cộng được, độc lập nhau**.

**Insight 3 — Mô tả đầy đủ tập đạt được.** Mọi chuỗi đạt được có dạng: "lấy `s`, cộng `x·a` vào nhóm chỉ số **gốc** lẻ, cộng `y·a` vào nhóm gốc chẵn, rồi xoay `r`" với `r` là bội của `g`, `x ∈ 0..9`, `y ∈ 0..9` nếu `b` lẻ (ngược lại `y = 0`). Ngược lại mọi bộ như vậy đều đạt được. ⇒ Liệt kê `(r, x, y)`: tối đa `n · 100` chuỗi.

**Insight 4 — Tham lam theo chữ số đầu.** Cố định khung xoay `t`. Chữ số ở vị trí 0 chỉ phụ thuộc lượng cộng nhóm chẵn, vị trí 1 chỉ phụ thuộc nhóm lẻ. Vì thứ tự từ điển xét vị trí 0 trước:
- **nếu `b` lẻ thì** chọn lượng cộng nhóm chẵn làm `t[0]` nhỏ nhất;
- chọn lượng cộng nhóm lẻ làm `t[1]` nhỏ nhất.

Lượng cộng làm nhỏ nhất `t[1]` là **duy nhất** modulo 10 (cùng một chữ số đích ⇒ cùng một lượng cộng), nên lựa chọn này cố định toàn bộ nhóm lẻ ⇒ mỗi khung chỉ còn **một** ứng viên. Thời gian `O(n/g · (10 + n))`.

In [10]:
# Kiểm chứng Insight 1–3 bằng BFS: tập đạt được == tập liệt kê (r, x, y)
def reachable_enum(s, a, b):
    n = len(s); g = gcd(b, n); res = set()
    ys = range(10) if b % 2 else [0]
    for x in range(10):
        for y in ys:
            base = "".join(str((int(c) + (x if i % 2 else y) * a) % 10) for i, c in enumerate(s))
            for r in range(0, n, g):
                res.add(rotate(base, r))
    return res

rnd = random.Random(0)
for _ in range(300):
    n = rnd.choice([2, 4, 6, 8, 10, 12])
    s = "".join(rnd.choice("0123456789") for _ in range(n))
    a = rnd.randint(1, 9); b = rnd.randint(1, n - 1)
    assert reachable_bfs(s, a, b) == reachable_enum(s, a, b), (s, a, b)
print("Tập đạt được khớp mô tả (r, x, y) trên 300 input ngẫu nhiên")

Tập đạt được khớp mô tả (r, x, y) trên 300 input ngẫu nhiên


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Liệt kê trực tiếp `(độ xoay, cộng lẻ, cộng chẵn)`

Từng đoạn:
1. `g = gcd(b, n)`: các độ xoay `0, g, 2g, …`.
2. Với mỗi khung xoay `t`, với mỗi `x ∈ 0..9` (cộng `x·a` vào các **vị trí lẻ của khung**) và `y ∈ 0..9` nếu `b` lẻ (cộng vào **vị trí chẵn**), dựng chuỗi và cập nhật `best`.

(Cộng trên khung hay trên chuỗi gốc đều như nhau vì ở mỗi khung, hai nhóm vị trí vẫn là hai nhóm gốc, có thể đã đổi vai — và khi `b` lẻ thì cả hai nhóm đều chạy hết `0..9`.)

In [11]:
def smallest_enum(s, a, b):
    n = len(s); g = gcd(b, n)
    ys = range(10) if b % 2 else [0]
    best = s
    for r in range(0, n, g):
        t = rotate(s, r)
        digits = [int(c) for c in t]
        for x in range(10):
            for y in ys:
                cand = "".join(str((d + (x if i % 2 else y) * a) % 10) for i, d in enumerate(digits))
                if cand < best:
                    best = cand
    return best

for (s, a, b), want in examples:
    print((s, a, b), smallest_enum(s, a, b), "kỳ vọng", want)

('5525', 9, 2) 2050 kỳ vọng 2050
('74', 5, 1) 24 kỳ vọng 24
('0011', 4, 2) 0011 kỳ vọng 0011


Code nộp LeetCode của Cách 2:

```python
from math import gcd

class Solution:
    def findLexSmallestString(self, s: str, a: int, b: int) -> str:
        n = len(s)
        g = gcd(b, n)
        ys = range(10) if b % 2 else [0]
        best = s
        for r in range(0, n, g):
            t = s[-r:] + s[:-r] if r else s
            digits = [int(c) for c in t]
            for x in range(10):
                for y in ys:
                    cand = "".join(str((d + (x if i % 2 else y) * a) % 10)
                                   for i, d in enumerate(digits))
                    best = min(best, cand)
        return best
```

**Độ phức tạp:** Time `O((n/g) · 100 · n)` ≤ `10⁶`, Space `O(n)` — không cần `visited`.

### 7.2 Cách 3 — Tham lam theo hai chữ số đầu

Cho khung `t`:
1. `oddAdd` = lượng cộng `k·a mod 10` làm `(t[1] + oddAdd) mod 10` nhỏ nhất.
2. Nếu `b` lẻ: `evenAdd` = lượng cộng làm `(t[0] + evenAdd) mod 10` nhỏ nhất; ngược lại `evenAdd = 0`.
3. Dựng một ứng viên duy nhất, so với `best`.

Hàm phụ `best_shift(d, a)` thử `k = 0..9` và trả lượng cộng tốt nhất cho chữ số `d`.

In [12]:
def best_shift(d, a):
    """Lượng cộng (k*a mod 10) làm (d + lượng) mod 10 nhỏ nhất."""
    return min((k * a % 10 for k in range(10)), key=lambda add: (d + add) % 10)

def smallest_greedy(s, a, b):
    n = len(s); g = gcd(b, n)
    best = s
    for r in range(0, n, g):
        t = rotate(s, r)
        odd_add = best_shift(int(t[1]), a)
        even_add = best_shift(int(t[0]), a) if b % 2 else 0
        cand = "".join(str((int(c) + (odd_add if i % 2 else even_add)) % 10) for i, c in enumerate(t))
        if cand < best:
            best = cand
    return best

for (s, a, b), want in examples:
    print((s, a, b), smallest_greedy(s, a, b), "kỳ vọng", want)

('5525', 9, 2) 2050 kỳ vọng 2050
('74', 5, 1) 24 kỳ vọng 24
('0011', 4, 2) 0011 kỳ vọng 0011


Vì sao chọn theo `t[1]` là đủ cho cả nhóm lẻ? Hai lượng cộng `u ≠ v` (mod 10) cho `(t[1] + u) mod 10 ≠ (t[1] + v) mod 10` ⇒ chữ số nhỏ nhất ở vị trí 1 ứng với **đúng một** lượng cộng. Mọi lựa chọn khác làm vị trí 1 lớn hơn ⇒ chuỗi lớn hơn bất kể phần sau. (Vị trí 0 được quyết định trước, độc lập với nhóm lẻ.)

**Độ phức tạp:** Time `O((n/g) · (20 + n))` = `O(n²/g)`, Space `O(n)`.

### 7.3 Cách 4 — DFS (stack) trên trạng thái nén `(r, x, y)`

Một cách "BFS/DFS thông minh hơn": không lưu chuỗi trong `visited` mà lưu bộ ba **nhỏ** `(độ xoay r, lượng cộng nhóm gốc lẻ x, lượng cộng nhóm gốc chẵn y)`. Hai phép toán biến đổi bộ ba:
- **Cộng:** vị trí lẻ hiện tại chứa ký tự gốc lẻ nếu `r` chẵn, ký tự gốc chẵn nếu `r` lẻ ⇒ tăng `x` hoặc `y` thêm `a` (mod 10).
- **Xoay:** `r ← (r + b) mod n`.

Cách này giữ nguyên tinh thần "duyệt đồ thị" của Cách 1 nhưng `visited` chỉ chứa tối đa `100n` bộ ba, chuỗi chỉ dựng khi cần so sánh. Hữu ích khi bạn chưa chứng minh được Insight 3 nhưng muốn tiết kiệm bộ nhớ.

In [13]:
def smallest_dfs_state(s, a, b):
    n = len(s)
    def build(r, x, y):
        base = "".join(str((int(c) + (x if i % 2 else y)) % 10) for i, c in enumerate(s))
        return rotate(base, r)
    start = (0, 0, 0)
    seen, stack = {start}, [start]
    best = s
    while stack:
        r, x, y = stack.pop()
        cand = build(r, x, y)
        if cand < best:
            best = cand
        if r % 2 == 0:
            nxt_add = (r, (x + a) % 10, y)
        else:
            nxt_add = (r, x, (y + a) % 10)
        nxt_rot = ((r + b) % n, x, y)
        for st in (nxt_add, nxt_rot):
            if st not in seen:
                seen.add(st); stack.append(st)
    return best

for (s, a, b), want in examples:
    print((s, a, b), smallest_dfs_state(s, a, b), "kỳ vọng", want)

('5525', 9, 2) 2050 kỳ vọng 2050
('74', 5, 1) 24 kỳ vọng 24
('0011', 4, 2) 0011 kỳ vọng 0011


Code nộp LeetCode của Cách 4:

```python
class Solution:
    def findLexSmallestString(self, s: str, a: int, b: int) -> str:
        n = len(s)
        def build(r, x, y):
            base = "".join(str((int(c) + (x if i % 2 else y)) % 10) for i, c in enumerate(s))
            return base[-r:] + base[:-r] if r else base
        seen, stack, best = {(0, 0, 0)}, [(0, 0, 0)], s
        while stack:
            r, x, y = stack.pop()
            best = min(best, build(r, x, y))
            add = (r, (x + a) % 10, y) if r % 2 == 0 else (r, x, (y + a) % 10)
            for st in (add, ((r + b) % n, x, y)):
                if st not in seen:
                    seen.add(st); stack.append(st)
        return best
```

**Độ phức tạp:** Time `O(100n · n)`, Space `O(100n)` bộ ba (thay vì `100n` chuỗi dài `n`).

## 8. TRACE từng bước

Trace **Cách 3** trên `s = "43987654", a = 7, b = 3` (ví dụ tự dựng — đối chiếu lại đề gốc; đáp án đã kiểm chứng bằng BFS là `"00553311"`). `n = 8`, `g = gcd(3, 8) = 1` ⇒ cả 8 khung; `b` lẻ ⇒ cộng được cả hai nhóm. Với `a = 7`, `k·7 mod 10` chạy qua mọi giá trị `0..9` ⇒ luôn đưa được `t[0]` và `t[1]` về 0.

| r | khung t | t[0] → even_add | t[1] → odd_add | ứng viên |
|---|---|---|---|---|
| 0 | 43987654 | 4 → +6 | 3 → +7 | 00553311 |
| 1 | 44398765 | 4 → +6 | 4 → +6 | 00954321 |
| … | … | … | … | … |

Hàm trace in đủ 8 dòng:

In [14]:
def trace_greedy(s, a, b):
    n = len(s); g = gcd(b, n); best = s
    print(f"n={n}, g=gcd({b},{n})={g}, b {'lẻ -> cộng được nhóm chẵn' if b % 2 else 'chẵn -> nhóm chẵn cố định'}")
    for r in range(0, n, g):
        t = rotate(s, r)
        odd_add = best_shift(int(t[1]), a)
        even_add = best_shift(int(t[0]), a) if b % 2 else 0
        cand = "".join(str((int(c) + (odd_add if i % 2 else even_add)) % 10) for i, c in enumerate(t))
        mark = ""
        if cand < best:
            best = cand; mark = "  <- best mới"
        print(f"r={r}: t={t}  even_add={even_add}  odd_add={odd_add}  cand={cand}{mark}")
    print("kết quả:", best)
    return best

assert trace_greedy("43987654", 7, 3) == smallest_bfs("43987654", 7, 3) == "00553311"

n=8, g=gcd(3,8)=1, b lẻ -> cộng được nhóm chẵn
r=0: t=43987654  even_add=6  odd_add=7  cand=00553311  <- best mới
r=1: t=44398765  even_add=6  odd_add=6  cand=00954321
r=2: t=54439876  even_add=5  odd_add=6  cand=00994422
r=3: t=65443987  even_add=4  odd_add=5  cand=00897422
r=4: t=76544398  even_add=3  odd_add=4  cand=00887722
r=5: t=87654439  even_add=2  odd_add=3  cand=00886752
r=6: t=98765443  even_add=1  odd_add=2  cand=00886655
r=7: t=39876544  even_add=7  odd_add=1  cand=00583615
kết quả: 00553311


In [15]:
trace_greedy("5525", 9, 2)

n=4, g=gcd(2,4)=2, b chẵn -> nhóm chẵn cố định
r=0: t=5525  even_add=0  odd_add=5  cand=5020  <- best mới
r=2: t=2555  even_add=0  odd_add=5  cand=2050  <- best mới
kết quả: 2050


'2050'

Trace **Cách 1 (BFS)** cho ví dụ 2: in từng trạng thái lấy ra khỏi hàng đợi cùng hai trạng thái kề.

In [16]:
def trace_bfs(s, a, b, limit=12):
    seen, q, step = {s}, deque([s]), 0
    while q:
        x = q.popleft()
        nxt = (add_odd(x, a), rotate(x, b))
        new = [y for y in nxt if y not in seen]
        for y in new:
            seen.add(y); q.append(y)
        if step < limit:
            print(f"pop {x}: cộng->{nxt[0]}, xoay->{nxt[1]}, mới={new}")
        step += 1
    print(f"tổng {len(seen)} trạng thái, nhỏ nhất = {min(seen)}")
    return min(seen)

assert trace_bfs("74", 5, 1) == "24"

pop 74: cộng->79, xoay->47, mới=['79', '47']
pop 79: cộng->74, xoay->97, mới=['97']
pop 47: cộng->42, xoay->74, mới=['42']
pop 97: cộng->92, xoay->79, mới=['92']
pop 42: cộng->47, xoay->24, mới=['24']
pop 92: cộng->97, xoay->29, mới=['29']
pop 24: cộng->29, xoay->42, mới=[]
pop 29: cộng->24, xoay->92, mới=[]
tổng 8 trạng thái, nhỏ nhất = 24


## 9. Code nộp LeetCode

Dùng Cách 3 (tham lam) — nhanh nhất và ngắn.

In [17]:
class Solution:
    def findLexSmallestString(self, s: str, a: int, b: int) -> str:
        n = len(s)
        g = gcd(b, n)
        shifts = [k * a % 10 for k in range(10)]

        def best_add(d):
            return min(shifts, key=lambda add: (d + add) % 10)

        best = s
        for r in range(0, n, g):
            t = s[-r:] + s[:-r] if r else s
            odd_add = best_add(int(t[1]))
            even_add = best_add(int(t[0])) if b % 2 else 0
            cand = "".join(str((int(c) + (odd_add if i % 2 else even_add)) % 10)
                           for i, c in enumerate(t))
            if cand < best:
                best = cand
        return best

for (s, a, b), want in examples:
    got = Solution().findLexSmallestString(s, a, b)
    print((s, a, b), got, "OK" if got == want else "SAI")

('5525', 9, 2) 2050 OK
('74', 5, 1) 24 OK
('0011', 4, 2) 0011 OK


## 10. Vì sao thuật toán đúng

**(a) Tập đạt được = `{xoay_r(cộng(s, x, y))}`.** Gọi `E`, `O` là nhóm ký tự ở chỉ số chẵn / lẻ của chuỗi gốc. Phép xoay `r` (`n` chẵn) giữ nguyên hai nhóm nếu `r` chẵn và đổi vai nếu `r` lẻ; phép cộng tác động lên nhóm đang nằm ở vị trí lẻ. Do đó mọi chuỗi đạt được đều là "chuỗi gốc với nhóm `O` cộng `x`, nhóm `E` cộng `y`, xoay `r`" — đây là một **bất biến** được giữ qua mỗi phép toán.
- `r` luôn là tổng các bước `b` ⇒ là bội của `g = gcd(b, n)`; và mọi bội của `g` đều đạt được (Bezout).
- Nếu `b` chẵn: mọi `r` chẵn ⇒ phép cộng luôn rơi vào `O` ⇒ `y = 0`.
- Nếu `b` lẻ: `g` lẻ nên `r = g` là lẻ; xoay `g`, cộng (nhóm `E` nhận `+a`), rồi xoay thêm `n − g` (cũng là bội của `g`) để về khung cũ ⇒ tăng riêng `y`. Tương tự tăng riêng `x` ở khung `r = 0`. Vậy mọi `(r, x, y)` đều đạt được.

**(b) Tham lam.** Trong một khung cố định, vị trí 0 chỉ phụ thuộc lượng cộng nhóm ở vị trí chẵn, vị trí 1 chỉ phụ thuộc nhóm ở vị trí lẻ. Ứng viên tối ưu của khung phải có vị trí 0 nhỏ nhất (nếu điều khiển được), rồi vị trí 1 nhỏ nhất; mỗi lựa chọn đó xác định **duy nhất** lượng cộng (mục 7.2) ⇒ toàn bộ chuỗi được xác định. Lấy min qua mọi khung ⇒ đáp án.

Mục 6 đã kiểm chứng (a) bằng BFS trên 300 input ngẫu nhiên; mục 12 kiểm chứng (b).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s) ≤ 100`, `g = gcd(b, n)`, số khung `m = n / g ≤ n`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. BFS trên chuỗi | `O(100 · n · n)` | `O(100 · n · n)` | lưu tới `10⁴` chuỗi dài 100 |
| 2. Liệt kê `(r, x, y)` | `O(m · 100 · n)` | `O(n)` | không cần `visited` |
| 3. Tham lam | `O(m · (20 + n))` | `O(n)` | nhanh nhất |
| 4. DFS trạng thái nén | `O(100 · n · n)` | `O(100 · n)` | bộ ba thay vì chuỗi |

Số trạng thái thực tế: `m · c · (c nếu b lẻ, 1 nếu b chẵn)` với `c = 10 / gcd(a, 10)` — trường hợp xấu nhất `n · 100` khi `g = 1`, `gcd(a,10) = 1`.

In [18]:
rng = random.Random(11)
big = "".join(rng.choice("0123456789") for _ in range(100))
for f in (smallest_bfs, smallest_enum, smallest_greedy, smallest_dfs_state):
    t = time.perf_counter()
    v = f(big, 7, 1)
    print(f"{f.__name__:20s} n=100, a=7, b=1 -> {v[:16]}... {time.perf_counter() - t:.3f}s")

smallest_bfs         n=100, a=7, b=1 -> 0004636905427116... 0.084s


smallest_enum        n=100, a=7, b=1 -> 0004636905427116... 0.080s
smallest_greedy      n=100, a=7, b=1 -> 0004636905427116... 0.001s


smallest_dfs_state   n=100, a=7, b=1 -> 0004636905427116... 0.114s


## 12. Bộ test

- Ví dụ đề + ví dụ tự dựng `("43987654", 7, 3) → "00553311"` (kiểm chứng bằng BFS; đối chiếu lại đề gốc).
- Edge: `n = 2`; `a = 5` (chỉ 2 độ cộng); `a` chẵn (5 độ cộng); `b` chẵn với `gcd(b, n) > 1`; chuỗi toàn `0`; chuỗi có chu kỳ (nhiều khung trùng nhau).
- Ngẫu nhiên seed cố định, đúng ràng buộc (`n` chẵn `2..100`, `1 ≤ a ≤ 9`, `1 ≤ b ≤ n − 1`), so mọi cách với BFS.

In [19]:
ALL = [smallest_bfs, smallest_enum, smallest_greedy, smallest_dfs_state,
       lambda s, a, b: Solution().findLexSmallestString(s, a, b)]
fixed = examples + [
    (("43987654", 7, 3), "00553311"),
    (("00", 1, 1), "00"),
    (("19", 1, 1), "00"),
    (("98", 2, 1), "01"),
    (("3456", 5, 2), "3456"),
    (("1234", 1, 2), "1032"),
    (("0000", 3, 1), "0000"),
]
for (s, a, b), want in fixed:
    for f in ALL:
        assert f(s, a, b) == want, (f, s, a, b)
print("ví dụ cố định: OK")

ví dụ cố định: OK


In [20]:
rnd = random.Random(1625)
for _ in range(400):
    n = rnd.choice(range(2, 31, 2))
    s = "".join(rnd.choice("0123456789") for _ in range(n))
    a = rnd.randint(1, 9); b = rnd.randint(1, n - 1)
    want = smallest_bfs(s, a, b)
    for f in ALL[1:]:
        assert f(s, a, b) == want, (f, s, a, b)
for _ in range(15):                                   # n lớn, đúng ràng buộc
    n = rnd.choice(range(60, 101, 2))
    s = "".join(rnd.choice("0123456789") for _ in range(n))
    a = rnd.randint(1, 9); b = rnd.randint(1, n - 1)
    want = smallest_greedy(s, a, b)
    assert smallest_bfs(s, a, b) == want == Solution().findLexSmallestString(s, a, b)
print("random tests: OK")

random tests: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ thử xoay `0` và `b` (không lặp phép xoay)

In [21]:
def wrong_rotate_once(s, a, b):
    best = s
    ys = range(10) if b % 2 else [0]
    for t in (s, rotate(s, b)):                       # SAI: thiếu 2b, 3b, ...
        for x in range(10):
            for y in ys:
                cand = "".join(str((int(c) + (x if i % 2 else y) * a) % 10) for i, c in enumerate(t))
                best = min(best, cand)
    return best

cx = ("513297", 1, 2)
print("sai:", wrong_rotate_once(*cx), "| đúng:", smallest_bfs(*cx))

sai: 503196 | đúng: 309559


**Vì sao sai:** xoay được **bao nhiêu lần cũng được**; với `n = 6, b = 2` các khung là xoay `0, 2, 4`. Khung xoay 4 (`"329751"`) bắt đầu bằng 3 — nhỏ hơn mọi khung bản sai xét. **Sửa:** duyệt mọi bội của `gcd(b, n)`.

### Lỗi 2 — Quên rằng khi `b` lẻ thì nhóm chẵn cũng cộng được

In [22]:
def wrong_no_even(s, a, b):
    n = len(s); g = gcd(b, n); best = s
    for r in range(0, n, g):
        t = rotate(s, r)
        odd_add = best_shift(int(t[1]), a)                  # SAI: bỏ qua even_add khi b lẻ
        cand = "".join(str((int(c) + (odd_add if i % 2 else 0)) % 10) for i, c in enumerate(t))
        best = min(best, cand)
    return best

print("sai:", wrong_no_even("74", 5, 1), "| đúng:", smallest_bfs("74", 5, 1))

sai: 42 | đúng: 24


**Vì sao sai:** với `b` lẻ, xoay một lần đưa nhóm chẵn sang vị trí lẻ, cộng xong xoay về ⇒ nhóm chẵn cũng đổi được. Ví dụ 2 của đề đòi đổi chữ số đầu `7 → 2`. **Sửa:** nếu `b % 2 == 1` thì chọn thêm `even_add`.

### Lỗi 3 — Thử thiếu số lần cộng (`range(9)` thay vì `range(10)`)

In [23]:
def wrong_range9(s, a, b):
    n = len(s); g = gcd(b, n); best = s
    for r in range(0, n, g):
        t = rotate(s, r)
        for x in range(9):                                   # SAI: thiếu x = 9
            cand = "".join(str((int(c) + (x * a if i % 2 else 0)) % 10) for i, c in enumerate(t))
            best = min(best, cand)
    return best

print("sai:", wrong_range9("0101", 1, 2), "| đúng:", smallest_bfs("0101", 1, 2))

sai: 0101 | đúng: 0000


**Vì sao sai:** với `a = 1`, đưa chữ số `1` về `0` cần cộng 9 lần. Lượng cộng `k·a mod 10` có thể cần `k` tới 9. **Sửa:** luôn thử `k = 0..9` (hoặc lặp tới khi quay lại giá trị ban đầu).

### Lỗi 4 — BFS không có `visited` (lặp vô hạn)

In [24]:
def wrong_no_visited(s, a, b, cap=20000):
    q, best, steps = deque([s]), s, 0
    while q and steps < cap:              # 'cap' chỉ để demo dừng được; bản sai thật sự chạy mãi
        x = q.popleft(); steps += 1
        best = min(best, x)
        q.append(add_odd(x, a)); q.append(rotate(x, b))     # SAI: không kiểm tra đã thăm
    return steps, len(q)

steps, pending = wrong_no_visited("5525", 9, 2)
print(f"sau {steps} bước vẫn còn {pending} phần tử chờ (thực tế chỉ có {len(reachable_bfs('5525', 9, 2))} trạng thái)")

sau 20000 bước vẫn còn 20001 phần tử chờ (thực tế chỉ có 20 trạng thái)


**Vì sao sai:** đồ thị có chu trình (cộng 10 lần quay về chỗ cũ), không có `visited` thì hàng đợi tăng gấp đôi mỗi tầng và không bao giờ rỗng. **Sửa:** `seen = {s}` và chỉ đẩy trạng thái chưa thấy.

### Lỗi 5 — Cộng nhầm vào chỉ số chẵn (hiểu "odd" theo đánh số từ 1)

In [25]:
def add_even_wrong(s, a):
    return "".join(str((int(c) + a) % 10) if i % 2 == 0 else c for i, c in enumerate(s))   # SAI

def wrong_index(s, a, b):
    seen, q = {s}, deque([s])
    while q:
        x = q.popleft()
        for y in (add_even_wrong(x, a), rotate(x, b)):
            if y not in seen:
                seen.add(y); q.append(y)
    return min(seen)

print("sai:", wrong_index("5525", 9, 2), "| đúng:", smallest_bfs("5525", 9, 2))
print("ví dụ của đề: add_odd('3456', 5) =", add_odd("3456", 5), "| bản sai:", add_even_wrong("3456", 5))

sai: 0535 | đúng: 2050
ví dụ của đề: add_odd('3456', 5) = 3951 | bản sai: 8406


**Vì sao sai:** đề đánh số **từ 0**; "odd indices" là `1, 3, 5, …`. Kiểm tra ngay bằng ví dụ trong đề: `"3456"` cộng 5 phải ra `"3951"`. **Sửa:** điều kiện `i % 2 == 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Áp dụng các phép **bao nhiêu lần tùy ý**, tìm kết quả nhỏ nhất/có đạt được không" + không gian trạng thái nhỏ ⇒ **BFS/DFS + visited** là lời giải an toàn đầu tiên. Ước lượng số trạng thái trước: ở đây `≤ n · 10 · 10`.
- Phép xoay lặp lại ⇒ nghĩ đến **`gcd(b, n)`**.
- Phép cộng modulo 10 lặp lại ⇒ chỉ **10** (thực ra `10/gcd(a,10)`) giá trị.
- Có "chỉ số chẵn / lẻ" + phép dịch ⇒ xét **tính chẵn lẻ** của độ dịch.
- "Nhỏ nhất theo thứ tự từ điển" ⇒ tham lam theo **ký tự đầu tiên**, rồi ký tự kế tiếp…
- Khi có lời giải BFS chạy được, dùng nó làm **oracle** để kiểm lời giải tham lam.

## 15. Biến thể của bài

1. **Đếm số chuỗi phân biệt đạt được** (bài tập 5) — liệt kê `(r, x, y)` rồi bỏ trùng.
2. **Chuỗi lớn nhất** theo thứ tự từ điển — đổi `min` thành `max`, tham lam chọn chữ số lớn nhất.
3. **Kiểm tra chuỗi `t` có đạt được từ `s` không** — chỉ cần thử `n/g` khung và kiểm tra hiệu chữ số từng nhóm là hằng số thuộc `{k·a mod 10}`.
4. **LC 899 Orderly Queue** — xoay/đưa ký tự về cuối, tìm chuỗi nhỏ nhất (phân tích `k = 1` vs `k ≥ 2`).
5. **LC 752 Open the Lock** — BFS trên chuỗi chữ số với phép cộng/trừ modulo 10.
6. **LC 1585 / LC 796 Rotate String** — xoay chuỗi, so khớp.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS chuỗi | duyệt mọi chuỗi đạt được | `O(100n²)` | `O(100n²)` | lời giải đầu tiên, làm oracle |
| 2. Liệt kê `(r,x,y)` | mô tả đầy đủ tập đạt được | `O(m·100·n)` | `O(n)` | khi đã hiểu gcd + chẵn lẻ |
| 3. Tham lam | mỗi khung chỉ 1 ứng viên | `O(m·n)` | `O(n)` | lời giải tối ưu, ngắn |
| 4. DFS trạng thái nén | visited là bộ ba nhỏ | `O(100n²)` | `O(100n)` | khi bộ nhớ chặt mà chưa chứng minh được cấu trúc |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `("0000", 3, 1)`, `("19", 1, 1)`, `("3456", 5, 2)`, `("98", 2, 1)`.

**Bài 2 (tính tay):** với `s = "1234", a = 1, b = 2`: liệt kê các khung xoay, nhóm nào cộng được, và đáp án.

**Bài 3 (điền TODO):** viết `frames(s, b)` trả về danh sách các khung xoay đạt được (theo thứ tự `r = 0, g, 2g, …`).

**Bài 4 (tự viết lại):** viết lại Cách 3 mà **không** dùng `min(..., key=...)`: tự tìm lượng cộng bằng vòng lặp.

**Bài 5 (mini challenge):** viết `count_reachable(s, a, b)` đếm số chuỗi **phân biệt** đạt được mà không chạy BFS (liệt kê theo Insight 3), đối chiếu với BFS.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def frames(s, b):
    n = len(s)
    # TODO: g = gcd(b, n); trả [rotate(s, r) for r in 0, g, 2g, ...]
    pass

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def greedy_no_key(s, a, b):
    # TODO
    pass

def count_reachable(s, a, b):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tính `g = gcd(b, n)` và tính chẵn lẻ của `b`. Mức 2: `a = 1` hoặc `a = 3` đưa được mọi chữ số về 0; `a = 5` chỉ có `+0/+5`; `a = 2` chỉ có độ cộng chẵn. Mức 3: `"0000"`, `"00"`, `"3456"`, `"01"`.

**Bài 2.** Mức 1: `g = gcd(2, 4) = 2` ⇒ khung `"1234"`, `"3412"`. Mức 2: `b` chẵn ⇒ chỉ nhóm lẻ cộng được; khung `"1234"` thắng nhờ chữ số đầu 1. Mức 3: đưa `2 → 0` cần `+8` ⇒ `"1032"`.

**Bài 3.** Mức 1: dùng `gcd`. Mức 2: `range(0, n, g)`. Mức 3: nhớ trường hợp `r = 0` (`s[-0:]` là cả chuỗi!).

**Bài 4.** Mức 1: lặp `k = 0..9`, giữ `best_digit` và `best_add`. Mức 2: cập nhật khi `(d + k·a) % 10 < best_digit`. Mức 3: làm cho cả `t[0]` (nếu `b` lẻ) và `t[1]`.

**Bài 5.** Mức 1: ba vòng lặp `r`, `x`, `y`. Mức 2: cho vào `set` để bỏ trùng (chuỗi tuần hoàn có nhiều bộ cho cùng chuỗi). Mức 3: `x, y` chỉ cần chạy trên các **độ cộng phân biệt** `{k·a mod 10}`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2
b1 = [Solution().findLexSmallestString(*t) for t in [("0000", 3, 1), ("19", 1, 1), ("3456", 5, 2), ("98", 2, 1)]]
print("Bài 1:", b1)
assert b1 == ["0000", "00", "3456", "01"]
print("Bài 2 khung:", [rotate("1234", r) for r in range(0, 4, gcd(2, 4))])
assert trace_greedy("1234", 1, 2) == "1032"

Bài 1: ['0000', '00', '3456', '01']
Bài 2 khung: ['1234', '3412']
n=4, g=gcd(2,4)=2, b chẵn -> nhóm chẵn cố định
r=0: t=1234  even_add=0  odd_add=8  cand=1032  <- best mới
r=2: t=3412  even_add=0  odd_add=6  cand=3018
kết quả: 1032


In [29]:
# Bài 3
def frames_sol(s, b):
    n = len(s); g = gcd(b, n)
    return [rotate(s, r) for r in range(0, n, g)]

assert frames_sol("3456", 2) == ["3456", "5634"]
assert len(frames_sol("12345678", 3)) == 8 and len(frames_sol("123456", 4)) == 3
rnd = random.Random(3)
for _ in range(200):
    n = rnd.choice(range(2, 21, 2)); s = "".join(rnd.choice("0123456789") for _ in range(n)); b = rnd.randint(1, n - 1)
    seen, x = set(), s
    for _ in range(n):
        seen.add(x); x = rotate(x, b)
    assert set(frames_sol(s, b)) == seen
print("Bài 3: OK")

Bài 3: OK


In [30]:
# Bài 4
def greedy_no_key_sol(s, a, b):
    n = len(s); g = gcd(b, n); best = s
    def pick(d):
        best_digit, best_add = 10, 0
        for k in range(10):
            v = (d + k * a) % 10
            if v < best_digit:
                best_digit, best_add = v, k * a % 10
        return best_add
    for r in range(0, n, g):
        t = rotate(s, r)
        oa = pick(int(t[1])); ea = pick(int(t[0])) if b % 2 else 0
        cand = "".join(str((int(c) + (oa if i % 2 else ea)) % 10) for i, c in enumerate(t))
        if cand < best:
            best = cand
    return best

rnd = random.Random(4)
for _ in range(300):
    n = rnd.choice(range(2, 41, 2)); s = "".join(rnd.choice("0123456789") for _ in range(n))
    a = rnd.randint(1, 9); b = rnd.randint(1, n - 1)
    assert greedy_no_key_sol(s, a, b) == smallest_bfs(s, a, b)
print("Bài 4: OK")

Bài 4: OK


In [31]:
# Bài 5
def count_reachable_sol(s, a, b):
    n = len(s); g = gcd(b, n)
    adds = sorted({k * a % 10 for k in range(10)})
    ys = adds if b % 2 else [0]
    res = set()
    for x in adds:
        for y in ys:
            base = "".join(str((int(c) + (x if i % 2 else y)) % 10) for i, c in enumerate(s))
            for r in range(0, n, g):
                res.add(rotate(base, r))
    return len(res)

assert count_reachable_sol("3456", 5, 2) == 4
assert count_reachable_sol("98", 2, 1) == 50
assert count_reachable_sol("0000", 3, 1) == 100       # chuỗi đều: nhiều khung trùng nhau
rnd = random.Random(5)
for _ in range(300):
    n = rnd.choice(range(2, 17, 2)); s = "".join(rnd.choice("0123") for _ in range(n))
    a = rnd.randint(1, 9); b = rnd.randint(1, n - 1)
    assert count_reachable_sol(s, a, b) == len(reachable_bfs(s, a, b))
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template BFS trạng thái (oracle):**

```python
seen, q = {s}, deque([s])
while q:
    x = q.popleft()
    for y in (op1(x), op2(x)):
        if y not in seen:
            seen.add(y); q.append(y)
return min(seen)
```

**Template tham lam của bài:**

```python
g = gcd(b, n)
for r in range(0, n, g):                 # mọi khung xoay
    t = s[-r:] + s[:-r] if r else s
    odd_add  = best_add(int(t[1]))        # luôn điều khiển được
    even_add = best_add(int(t[0])) if b % 2 else 0
    best = min(best, build(t, odd_add, even_add))
```

**Checklist:**
- [ ] "Odd indices" là `1, 3, 5, …` (đánh số từ 0)? Thử `"3456"`, `a = 5` → `"3951"`.
- [ ] Đã xét **mọi** độ xoay là bội của `gcd(b, n)` (không chỉ `0` và `b`)?
- [ ] `b` lẻ ⇒ nhóm chẵn **cũng** cộng được?
- [ ] Thử đủ `k = 0..9` lần cộng?
- [ ] `s[-0:]` là cả chuỗi — xử lý riêng `r = 0`?
- [ ] BFS có `visited`?